# Naive Baseline – 30‑Minute Waste Forecast

This notebook implements the **Naive** baseline: predict the last observed value for all future steps.

Saves results and plots to `deployment_models/`.

In [ ]:
# 1. Imports
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
from google.colab import drive

warnings.filterwarnings('ignore')
np.random.seed(42)

# 2. Mount drive & set working directory
drive.mount('/content/drive', force_remount=True)
try:
    os.chdir('/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence')
    print('Directory changed to project folder')
except OSError:
    print("Error: Could not change directory. Please check the path.")

In [ ]:
# 3. Configuration
DATA_PATH   = 'data/food_waste_augmented.csv'
MODEL_DIR   = 'deployment_models'
MODEL_NAME  = 'Naive'

FREQ        = '30min'
ALIGN_SECTIONS = 'union'
TRAIN_RATIO = 0.7
VAL_RATIO   = 0.1
TEST_RATIO  = 0.2
LOOKBACK    = 24
MIN_SAMPLES = 10

os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Aggregation frequency: {FREQ}")
print(f"Model: {MODEL_NAME}")

In [ ]:
# 4. Load and aggregate data
df = pd.read_csv(DATA_PATH, parse_dates=['Date'])
daily_section = (
    df.set_index('Date')
      .groupby([pd.Grouper(freq=FREQ), 'Canteen_Section'])['Waste_Weight_kg']
      .sum()
      .reset_index()
      .rename(columns={'Waste_Weight_kg': 'Total_Waste_kg'})
)
daily_wide = (
    daily_section
    .pivot(index='Date', columns='Canteen_Section', values='Total_Waste_kg')
    .sort_index()
)

if ALIGN_SECTIONS == 'common':
    daily_wide = daily_wide.dropna()
else:
    daily_wide = daily_wide.fillna(0)

sections = daily_wide.columns.tolist()
print(f"Sections: {sections}")

In [ ]:
# 5. Split (chronological)
ref_index = daily_wide.index
n_total   = len(ref_index)
n_test = max(1, int(n_total * TEST_RATIO))
n_val  = max(1, int(n_total * VAL_RATIO))
n_train = n_total - n_val - n_test

train_indices = ref_index[:n_train]
val_indices   = ref_index[n_train:n_train + n_val]
test_indices  = ref_index[n_train + n_val:]

train_mask = ref_index.isin(train_indices)
val_mask   = ref_index.isin(val_indices)
test_mask  = ref_index.isin(test_indices)

print(f"Train: {len(train_indices)}, Val: {len(val_indices)}, Test: {len(test_indices)}")

In [ ]:
# 6. Helper functions
def compute_regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else np.nan
    r2 = r2_score(y_true, y_pred)
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

def plot_predictions(y_true, y_pred, dates, section, max_points=200, save_path=None):
    if len(dates) > max_points:
        step = len(dates) // max_points
        idx = range(0, len(dates), step)
        dates = dates[idx]; y_true = y_true[idx]; y_pred = y_pred[idx]
    plt.figure(figsize=(12,5))
    plt.plot(dates, y_true, label='Actual', marker='o', markersize=3, linestyle='-', linewidth=1)
    plt.plot(dates, y_pred, label='Predicted', marker='x', markersize=3, linestyle='--', linewidth=1)
    plt.xlabel('Date'); plt.ylabel('Waste (kg)')
    plt.title(f'{MODEL_NAME} Predictions vs Actual - Section {section}')
    plt.legend(); plt.grid(True); plt.xticks(rotation=45); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 7. Main loop
all_metrics = []

for sec in sections:
    print(f"\n{'='*60}\nTraining {MODEL_NAME} for section {sec}\n{'='*60}")

    y_series = daily_wide[sec]
    y_train = y_series[train_mask]
    y_test  = y_series[test_mask]

    if len(y_train) == 0:
        print(f"  Skipping section {sec} – training set empty.")
        continue

    # Naive forecast: repeat last observed training value
    last_value = y_train.iloc[-1]
    test_pred = np.full(len(y_test), last_value)

    # Metrics
    metrics = compute_regression_metrics(y_test.values, test_pred)
    print("\n--- Performance ---")
    for k,v in metrics.items(): print(f"  {k}: {v:.4f}")

    all_metrics.append({'Section': sec, 'Model': MODEL_NAME, **metrics})

    # Save artefacts
    artifacts = {
        'section': sec,
        'model_name': MODEL_NAME,
        'last_value': last_value,
        'train_date_range': [train_indices.min().isoformat(), train_indices.max().isoformat()],
        'val_date_range': [val_indices.min().isoformat(), val_indices.max().isoformat()],
        'test_date_range': [test_indices.min().isoformat(), test_indices.max().isoformat()]
    }
    save_path = f'{MODEL_DIR}/section_{sec}_{MODEL_NAME}.joblib'
    joblib.dump(artifacts, save_path)
    print(f"Saved model to {save_path}")

    # Plot predictions
    plot_predictions(y_test.values, test_pred, test_indices, sec,
                     save_path=f'{MODEL_DIR}/section_{sec}_{MODEL_NAME}_predictions.png')

# Save summary
if all_metrics:
    summary_df = pd.DataFrame(all_metrics)
    summary_df.to_csv(f'{MODEL_DIR}/test_metrics_{MODEL_NAME}.csv', index=False)
    print("\nSummary saved.")
else:
    print("No metrics collected.")